In [ ]:

# OrchNAS Scalability Analysis on ImageNet-100

!pip -q install datasets timm torchmetrics

import os
import copy
import math
import random
import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import timm
from datasets import load_dataset
from torchvision import transforms
import matplotlib.pyplot as plt


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


NUM_CLASSES = 100
IMAGE_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 10


TRAIN_SUBSET_SIZE = 12000
VAL_SUBSET_SIZE = 2000


SCALABILITY_SERVICE_COUNTS = [50, 200, 500, 1000, 2500]

# Training configuration
ROUNDS = 3                  # increase to 100 for heavier runs
SERVICES_PER_ROUND = 10
LOCAL_EPOCHS = 20            # paper says 20, lowered for Colab
GLOBAL_LR = 1e-3
PERSONAL_LR = 1e-3
LAMBDA_ENERGY = 1e-10
MU = 1e-4
RHO = 1e-10
BETA = 1e-10



train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

test_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

class HFImageDataset(Dataset):
    def __init__(self, hf_ds, transform=None):
        self.ds = hf_ds
        self.transform = transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        sample = self.ds[idx]
        img = sample["image"].convert("RGB")
        y = int(sample["label"])
        if self.transform:
            img = self.transform(img)
        return img, y

print("Loading ImageNet-100 dataset...")
try:
    ds = load_dataset("clane9/imagenet-100")
except Exception as e:
    raise RuntimeError(
        "Could not load Hugging Face dataset 'clane9/imagenet-100'. "
        "Please replace it with your ImageNet-100 source."
    ) from e

train_raw = ds["train"]
val_raw = ds["validation"] if "validation" in ds else ds["test"]


train_indices = np.random.choice(len(train_raw), min(TRAIN_SUBSET_SIZE, len(train_raw)), replace=False)
val_indices = np.random.choice(len(val_raw), min(VAL_SUBSET_SIZE, len(val_raw)), replace=False)

train_ds = HFImageDataset(train_raw.select(train_indices.tolist()), transform=train_tfms)
val_ds = HFImageDataset(val_raw.select(val_indices.tolist()), transform=test_tfms)

global_val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print("Train subset:", len(train_ds))
print("Val subset:", len(val_ds))

def dirichlet_partition(dataset: Dataset, num_clients: int, alpha: float = 0.5, seed: int = 42):
    rng = np.random.default_rng(seed)

    # collect labels
    labels = []
    for i in range(len(dataset)):
        _, y = dataset[i]
        labels.append(y)
    labels = np.array(labels)

    class_indices = [np.where(labels == c)[0] for c in range(NUM_CLASSES)]
    client_indices = [[] for _ in range(num_clients)]

    for c in range(NUM_CLASSES):
        idxs = class_indices[c]
        rng.shuffle(idxs)
        proportions = rng.dirichlet(alpha=np.repeat(alpha, num_clients))
        split_points = (np.cumsum(proportions) * len(idxs)).astype(int)[:-1]
        splits = np.split(idxs, split_points)
        for client_id, split in enumerate(splits):
            client_indices[client_id].extend(split.tolist())

    return client_indices


SEARCH_SPACE = {
    "width_mult": [0.5, 0.75, 1.0],
    "depth_mult": [0.5, 0.75, 1.0],
    "kernel_size": [3, 5],
    "expansion": [3, 6],
}

def random_architecture():
    return {
        "width_mult": random.choice(SEARCH_SPACE["width_mult"]),
        "depth_mult": random.choice(SEARCH_SPACE["depth_mult"]),
        "kernel_size": random.choice(SEARCH_SPACE["kernel_size"]),
        "expansion": random.choice(SEARCH_SPACE["expansion"]),
    }

def arch_to_key(arch):
    return (arch["width_mult"], arch["depth_mult"], arch["kernel_size"], arch["expansion"])

def mutate_architecture(arch, p=0.6):
    new_arch = copy.deepcopy(arch)
    for k, vals in SEARCH_SPACE.items():
        if random.random() < p:
            new_arch[k] = random.choice(vals)
    return new_arch


class OrchMobileNet(nn.Module):
    """
    MobileNetV2-style backbone + personalised head.
    width_mult is directly supported by timm.
    depth_mult, kernel_size, expansion are approximated in the energy model
    and pruning logic for a Colab-friendly prototype.
    """
    def __init__(self, arch, num_classes=100):
        super().__init__()
        self.arch = copy.deepcopy(arch)
        self.backbone = timm.create_model(
            "mobilenetv2_100",
            pretrained=False,
            num_classes=0,
            global_pool="avg",
            width_mult=arch["width_mult"],
        )
        feat_dim = self.backbone.num_features
        self.personal_head = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        feat = self.backbone(x)
        return self.personal_head(feat)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)



def estimate_flops_m(arch):
    width_scale = arch["width_mult"] ** 2
    depth_scale = arch["depth_mult"]
    kernel_scale = 1.0 if arch["kernel_size"] == 3 else 1.20
    expansion_scale = 1.0 if arch["expansion"] == 3 else 1.15
    return BASE_FLOPS_MB * width_scale * depth_scale * kernel_scale * expansion_scale

def estimate_energy_j(arch, alpha_k, local_steps):
    flops_m = estimate_flops_m(arch)
    return alpha_k * flops_m * local_steps

@dataclass
class ServiceState:
    energy_budget: float
    compute_budget: float
    memory_budget: float
    alpha: float
    local_steps: int

def sample_resource_state(group: str):
    if group == "low":
        return ServiceState(
            energy_budget=0.9,
            compute_budget=220.0,
            memory_budget=2_500_000,
            alpha=0.0030,
            local_steps=LOCAL_EPOCHS * 50,
        )
    elif group == "medium":
        return ServiceState(
            energy_budget=1.2,
            compute_budget=300.0,
            memory_budget=3_500_000,
            alpha=0.0024,
            local_steps=LOCAL_EPOCHS * 50,
        )
    else:
        return ServiceState(
            energy_budget=1.6,
            compute_budget=380.0,
            memory_budget=5_000_000,
            alpha=0.0018,
            local_steps=LOCAL_EPOCHS * 50,
        )


@torch.no_grad()
def evaluate_model(model, loader, max_batches=None):
    model.eval()
    total = 0
    correct = 0
    for i, (x, y) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        pred = logits.argmax(dim=1)
        total += y.size(0)
        correct += (pred == y).sum().item()
    return 100.0 * correct / max(total, 1)


def simpler_architecture_moves(arch):
    moves = []

    if arch["width_mult"] > min(SEARCH_SPACE["width_mult"]):
        vals = SEARCH_SPACE["width_mult"]
        idx = vals.index(arch["width_mult"])
        a = copy.deepcopy(arch)
        a["width_mult"] = vals[idx - 1]
        moves.append(a)

    if arch["depth_mult"] > min(SEARCH_SPACE["depth_mult"]):
        vals = SEARCH_SPACE["depth_mult"]
        idx = vals.index(arch["depth_mult"])
        a = copy.deepcopy(arch)
        a["depth_mult"] = vals[idx - 1]
        moves.append(a)

    if arch["kernel_size"] > min(SEARCH_SPACE["kernel_size"]):
        a = copy.deepcopy(arch)
        a["kernel_size"] = 3
        moves.append(a)

    if arch["expansion"] > min(SEARCH_SPACE["expansion"]):
        a = copy.deepcopy(arch)
        a["expansion"] = 3
        moves.append(a)

    return moves

def is_feasible(arch, state):
    temp_model = OrchMobileNet(arch, num_classes=NUM_CLASSES).to(DEVICE)
    params = count_parameters(temp_model)
    flops_m = estimate_flops_m(arch)
    energy = estimate_energy_j(arch, state.alpha, state.local_steps)
    del temp_model
    torch.cuda.empty_cache()

    return (
        energy <= state.energy_budget and
        flops_m <= state.compute_budget and
        params <= state.memory_budget
    )

def greedy_prune_architecture(current_arch, state, beta=BETA):
    arch = copy.deepcopy(current_arch)

    while not is_feasible(arch, state):
        candidates = simpler_architecture_moves(arch)
        if not candidates:
            break

        base_energy = estimate_energy_j(arch, state.alpha, state.local_steps)

        best_score = float("inf")
        best_arch = arch

        for cand in candidates:
            cand_energy = estimate_energy_j(cand, state.alpha, state.local_steps)
            delta_energy = base_energy - cand_energy
            delta_acc_proxy = max(0.0, estimate_flops_m(arch) - estimate_flops_m(cand)) * 0.001
            score = delta_acc_proxy - beta * delta_energy

            if score < best_score:
                best_score = score
                best_arch = cand

        if arch_to_key(best_arch) == arch_to_key(arch):
            break
        arch = best_arch

    return arch


def train_shared_backbone(model, loader, lr=GLOBAL_LR, epochs=1, max_batches=None):
    optimizer = torch.optim.Adam(
        list(model.backbone.parameters()) + list(model.personal_head.parameters()),
        lr=lr
    )
    model.train()

    for _ in range(epochs):
        for i, (x, y) in enumerate(loader):
            if max_batches is not None and i >= max_batches:
                break
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()

def train_personal_head(model, loader, global_head_state, mu=MU, lr=PERSONAL_LR, epochs=1, max_batches=None):
    optimizer = torch.optim.Adam(model.personal_head.parameters(), lr=lr)
    model.train()

    for _ in range(epochs):
        for i, (x, y) in enumerate(loader):
            if max_batches is not None and i >= max_batches:
                break
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            ce = F.cross_entropy(logits, y)

            prox = 0.0
            for name, p in model.personal_head.named_parameters():
                prox = prox + torch.sum((p - global_head_state[name].to(DEVICE)) ** 2)

            loss = ce + 0.5 * mu * prox
            loss.backward()
            optimizer.step()


class EdgeService:
    def __init__(self, service_id, train_subset, val_subset, resource_group):
        self.service_id = service_id
        self.train_loader = DataLoader(
            train_subset, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=NUM_WORKERS, pin_memory=True
        )
        self.val_loader = DataLoader(
            val_subset, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=True
        )
        self.base_state = sample_resource_state(resource_group)
        self.dual_eta = 0.0
        self.personal_arch = random_architecture()

    def observe_state(self):
        s = copy.deepcopy(self.base_state)
        s.energy_budget *= random.uniform(0.95, 1.05)
        s.compute_budget *= random.uniform(0.95, 1.05)
        s.memory_budget *= random.uniform(0.95, 1.05)
        s.alpha *= random.uniform(0.98, 1.02)
        return s

    def score_candidates(self, candidate_pool):
        scores = []
        state = self.observe_state()

        for arch in candidate_pool:
            model = OrchMobileNet(arch, num_classes=NUM_CLASSES).to(DEVICE)
            acc = evaluate_model(model, self.val_loader, max_batches=2)
            energy = estimate_energy_j(arch, state.alpha, state.local_steps)
            score = acc - LAMBDA_ENERGY * energy
            scores.append(score)
            del model
            torch.cuda.empty_cache()

        return scores

    def local_update(self, global_arch, global_backbone_state, global_head_state):
        state = self.observe_state()
        local_arch = copy.deepcopy(global_arch)

        if not is_feasible(local_arch, state):
            local_arch = greedy_prune_architecture(local_arch, state, beta=BETA)

        local_model = OrchMobileNet(local_arch, num_classes=NUM_CLASSES).to(DEVICE)
        local_model.backbone.load_state_dict(global_backbone_state, strict=False)
        local_model.personal_head.load_state_dict(global_head_state, strict=False)

        train_shared_backbone(local_model, self.train_loader, lr=GLOBAL_LR, epochs=1, max_batches=5)
        train_personal_head(local_model, self.train_loader, global_head_state, mu=MU, lr=PERSONAL_LR, epochs=1, max_batches=5)

        local_energy = estimate_energy_j(local_arch, state.alpha, state.local_steps)

        # dual update
        self.dual_eta = max(0.0, self.dual_eta + RHO * (local_energy - state.energy_budget))
        self.personal_arch = local_arch

        n = len(self.train_loader.dataset)
        return {
            "backbone": copy.deepcopy(local_model.backbone.state_dict()),
            "head": copy.deepcopy(local_model.personal_head.state_dict()),
            "n": n,
            "arch": local_arch,
            "energy": local_energy,
            "model": local_model
        }


def fedavg_state_dicts(state_dicts, weights):
    total = float(sum(weights))
    out = copy.deepcopy(state_dicts[0])
    for k in out.keys():
        out[k] = sum(sd[k] * (w / total) for sd, w in zip(state_dicts, weights))
    return out


def build_services(num_services, alpha_dirichlet=0.5):
    partitions = dirichlet_partition(train_ds, num_services, alpha=alpha_dirichlet, seed=42)
    val_partitions = dirichlet_partition(val_ds, num_services, alpha=alpha_dirichlet, seed=123)

    services = []
    groups = ["low", "medium", "high"]

    for k in range(num_services):
        tr_idx = partitions[k]
        va_idx = val_partitions[k]

        # avoid empty client
        if len(tr_idx) < 10:
            extra = np.random.choice(len(train_ds), 20, replace=False).tolist()
            tr_idx.extend(extra)

        if len(va_idx) < 5:
            extra = np.random.choice(len(val_ds), 10, replace=False).tolist()
            va_idx.extend(extra)

        train_subset = Subset(train_ds, tr_idx)
        val_subset = Subset(val_ds, va_idx)
        resource_group = groups[k % 3]

        services.append(EdgeService(k, train_subset, val_subset, resource_group))

    return services


class OrchNASServer:
    def __init__(self, services):
        self.services = services
        self.global_arch = random_architecture()
        self.global_model = OrchMobileNet(self.global_arch, num_classes=NUM_CLASSES).to(DEVICE)
        self.candidate_pool_size = 6

    def evolve_candidate_pool(self):
        pool = [copy.deepcopy(self.global_arch)]
        seen = {arch_to_key(self.global_arch)}

        while len(pool) < self.candidate_pool_size:
            cand = mutate_architecture(self.global_arch, p=0.7)
            k = arch_to_key(cand)
            if k not in seen:
                seen.add(k)
                pool.append(cand)

        return pool

    def select_global_architecture(self, selected_services, candidate_pool):
        sizes = [len(s.train_loader.dataset) for s in selected_services]
        total = sum(sizes)
        weighted_scores = [0.0 for _ in candidate_pool]

        for service, size in zip(selected_services, sizes):
            pk = size / total
            scores = service.score_candidates(candidate_pool)
            for i, sc in enumerate(scores):
                weighted_scores[i] += pk * sc

        best_idx = int(np.argmax(weighted_scores))
        return candidate_pool[best_idx]

    def train(self, rounds=ROUNDS, services_per_round=SERVICES_PER_ROUND):
        for rnd in range(1, rounds + 1):
            selected = random.sample(self.services, min(services_per_round, len(self.services)))
            candidate_pool = self.evolve_candidate_pool()
            self.global_arch = self.select_global_architecture(selected, candidate_pool)

            old_backbone = copy.deepcopy(self.global_model.backbone.state_dict())
            old_head = copy.deepcopy(self.global_model.personal_head.state_dict())

            self.global_model = OrchMobileNet(self.global_arch, num_classes=NUM_CLASSES).to(DEVICE)
            self.global_model.backbone.load_state_dict(old_backbone, strict=False)
            self.global_model.personal_head.load_state_dict(old_head, strict=False)

            local_backbones, local_heads, local_ns = [], [], []
            local_energies = []
            personal_accs = []

            for service in selected:
                result = service.local_update(
                    self.global_arch,
                    copy.deepcopy(self.global_model.backbone.state_dict()),
                    copy.deepcopy(self.global_model.personal_head.state_dict())
                )
                local_backbones.append(result["backbone"])
                local_heads.append(result["head"])
                local_ns.append(result["n"])
                local_energies.append(result["energy"])

                acc = evaluate_model(result["model"], service.val_loader, max_batches=2)
                personal_accs.append(acc)
                del result["model"]
                torch.cuda.empty_cache()

            avg_backbone = fedavg_state_dicts(local_backbones, local_ns)
            avg_head = fedavg_state_dicts(local_heads, local_ns)

            self.global_model.backbone.load_state_dict(avg_backbone, strict=False)
            self.global_model.personal_head.load_state_dict(avg_head, strict=False)

            print(
                f"Round {rnd:02d} | "
                f"Global arch={self.global_arch} | "
                f"Avg personal acc={np.mean(personal_accs):.2f}% | "
                f"Avg energy={np.mean(local_energies):.4f} J"
            )

    def evaluate_personalised_accuracy(self, max_services_eval=50):
        sampled = random.sample(self.services, min(max_services_eval, len(self.services)))
        accs = []

        for s in sampled:
            local_model = OrchMobileNet(s.personal_arch, num_classes=NUM_CLASSES).to(DEVICE)
            local_model.backbone.load_state_dict(self.global_model.backbone.state_dict(), strict=False)
            local_model.personal_head.load_state_dict(self.global_model.personal_head.state_dict(), strict=False)
            acc = evaluate_model(local_model, s.val_loader, max_batches=2)
            accs.append(acc)
            del local_model
            torch.cuda.empty_cache()

        return float(np.mean(accs)) if accs else 0.0

    def estimate_avg_training_energy(self, max_services_eval=50):
        sampled = random.sample(self.services, min(max_services_eval, len(self.services)))
        energies = []
        for s in sampled:
            state = s.observe_state()
            energies.append(estimate_energy_j(s.personal_arch, state.alpha, state.local_steps))
        return float(np.mean(energies)) if energies else 0.0


results = {
    "num_services": [],
    "personalised_acc": [],
    "avg_energy": [],
}

for num_services in SCALABILITY_SERVICE_COUNTS:
    print("\n" + "=" * 70)
    print(f"Scalability run with {num_services} services")
    print("=" * 70)

    services = build_services(num_services=num_services, alpha_dirichlet=0.5)
    server = OrchNASServer(services)
    server.train(rounds=ROUNDS, services_per_round=min(SERVICES_PER_ROUND, num_services))

    personalised_acc = server.evaluate_personalised_accuracy(max_services_eval=50)
    avg_energy = server.estimate_avg_training_energy(max_services_eval=50)

    results["num_services"].append(num_services)
    results["personalised_acc"].append(personalised_acc)
    results["avg_energy"].append(avg_energy)


df = pd.DataFrame({
    "Num_Services": results["num_services"],
    "Accuracy (%)": results["personalised_acc"],
    "Energy (J)": results["avg_energy"]
})

df.to_csv("orchnas_scalability_results.csv", index=False)

